# Ledger retention & cohort analysis
Run top to bottom (Restart → Run All). `ledger_loader.py` must be in the same folder as this notebook.

## 1. Setup

In [36]:
from ledger_loader import get_final_ledger
import pandas as pd
import numpy as np
import os
import re
import json
import matplotlib as mpl
from IPython.display import display
from openpyxl.cell.rich_text import CellRichText, TextBlock
from openpyxl.cell.text import InlineFont
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

CLIENTS_CSV      = r"C:\Users\Administrator\Videos\Clients\clients (1).csv"
NOT_RETURNED_DIR = r"C:\Users\Administrator\Videos\Not Returned"
COHORT_DIR       = r"C:\Users\Administrator\Videos\co"

## 2. Load ledger and build monthly activity

In [23]:
ledger = get_final_ledger()

entry = ["BILL"]
led = ledger[ledger["LedgerType"].isin(entry)].copy()
led["Month"] = led["CreatedAt"].dt.to_period("M")

activity = led[["ClientID", "Month"]].drop_duplicates()
activity["Cohort"] = activity.groupby("ClientID")["Month"].transform("min")
activity["MonthIdx"] = (activity["Month"] - activity["Cohort"]).apply(lambda x: x.n)

this_month = pd.Timestamp.today().to_period("M")   # or activity["Month"].max() for the latest month in the data
prev_month = this_month - 1
last_month = activity["Month"].max()               # end month for blanking future cohort cells
max_idx = activity["MonthIdx"].max()               # same M-columns on every cohort table

print("Ledger runs to:", ledger["CreatedAt"].max())

c:\Users\Administrator\Videos\ledger_loader.py:44: DtypeWarning: Columns (3: MemberID) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Ledger runs to: 2026-10-22 23:59:57


## 3. Add lead source and migration status
Assumes `ARC Client ID` is a column in the clients file. If it comes from a different file, merge that file into `activity_source` here instead, before the `Client Status` line.

In [24]:
clients = pd.read_csv(CLIENTS_CSV)
source_col = "Source One"

Leads_Map = clients[["Client ID", source_col, "ARC Client ID"]]
activity_source = activity.merge(Leads_Map, left_on="ClientID", right_on="Client ID", how="left")

activity_source[source_col] = activity_source[source_col].fillna("Unknown")
activity_source["Client Status"] = np.select(
    [activity_source["Client ID"].isna(), activity_source["ARC Client ID"].notna()],
    ["Unmatched", "Migrated"],
    default="Fresh",
)
activity_source.head()

C:\Users\Administrator\AppData\Local\Temp\ipykernel_10476\1556379475.py:1: DtypeWarning: Columns (4: Client ID (manual), 23: Tags, 27: Language) have mixed types. Specify dtype option on import or set low_memory=False.
  clients = pd.read_csv(CLIENTS_CSV)


,ClientID,Month,Cohort,MonthIdx,Client ID,Source One,ARC Client ID,Client Status
0,P71057,2026-08,2026-01,7,P71057,Referral (HOT),NaN,Fresh
1,P62149,2026-08,2026-02,6,P62149,MAIN WEBSITE (HOT),707683.0,Migrated
2,M129746,2026-08,2026-06,2,M129746,MAIN WEBSITE (HOT),NaN,Fresh
3,S39238,2026-08,2025-11,9,S39238,MAIN WEBSITE (HOT),629838.0,Migrated
4,R32184,2026-08,2025-11,9,R32184,MAIN WEBSITE (HOT),827197.0,Migrated


In [25]:
# sanity checks - look at these before building cohorts
print("Duplicate Client IDs in Leads_Map:", Leads_Map["Client ID"].duplicated().sum())          # should be 0
print("Clients not found in clients file:", activity_source.loc[activity_source["Client ID"].isna(), "ClientID"].nunique())
print("Clients with more than one status:", (activity_source.groupby("ClientID")["Client Status"].nunique() > 1).sum())  # should be 0

Duplicate Client IDs in Leads_Map: 0
Clients not found in clients file: 50
Clients with more than one status: 0


## 4. Helper functions
All reusable functions live in this one cell.

In [26]:
cmap = mpl.colormaps["Blues"].copy()
cmap.set_bad("white")
STATUS_ORDER = ["Migrated", "Fresh", "Unmatched"]      # order of the small breakdown line


def build_cohort(act):
    """Plain cohort table (used for the retention summaries)."""
    counts = (act.groupby(["Cohort", "MonthIdx"])["ClientID"].nunique()
                 .unstack(fill_value=0)
                 .reindex(columns=range(max_idx + 1), fill_value=0))
    size = counts[0]
    pct = counts.div(size, axis=0) * 100
    for cohort in pct.index:
        pct.loc[cohort, pct.columns > (last_month - cohort).n] = np.nan
    active = counts.where(pct.notna())

    # all cohorts combined: clients still active / clients whose cohort has reached that month
    overall = active.sum() / pct.notna().mul(size, axis=0).sum() * 100

    table = (pct.map("{:.0f}%".format) + " (" + active.map("{:,.0f}".format) + ")").where(pct.notna(), "—")
    table.columns = [f"M{c}" for c in table.columns]
    table.insert(0, "Size", size)
    table.index = table.index.strftime("%b %Y")
    table.index.name = "Cohort"

    styled = table.style.background_gradient(
        cmap=cmap, subset=table.columns[1:], gmap=pct.to_numpy(), axis=None, vmin=0, vmax=100)
    return styled, overall


def split_cohort(act, split_col="Client Status"):
    """Cohort where each cell is '712 (100%)' plus a small 'Migrated: 25 · Fresh: 687' line."""
    counts = (act.groupby(["Cohort", "MonthIdx"])["ClientID"].nunique()
                 .unstack(fill_value=0)
                 .reindex(columns=range(max_idx + 1), fill_value=0))
    size = counts[0]
    pct = counts.div(size, axis=0) * 100
    for cohort in pct.index:
        pct.loc[cohort, pct.columns > (last_month - cohort).n] = np.nan

    split = act.groupby(["Cohort", "MonthIdx", split_col])["ClientID"].nunique()

    main = pd.DataFrame("", index=pct.index, columns=pct.columns)
    sub = main.copy()
    for (cohort, idx), p in pct.stack().dropna().items():
        main.loc[cohort, idx] = f"{counts.loc[cohort, idx]:,} ({p:.0f}%)"
        parts = [f"{s}: {split.get((cohort, idx, s), 0):,}" for s in STATUS_ORDER
                 if s != "Unmatched" or split.get((cohort, idx, s), 0)]
        sub.loc[cohort, idx] = " · ".join(parts)
    return {"main": main, "sub": sub, "pct": pct, "size": size}


def show_split(grid, caption=None):
    """Notebook view of a split cohort: big line + small breakdown line."""
    cells = (grid["main"] + '<br><span style="font-size:0.72em;opacity:0.85">'
             + grid["sub"] + "</span>").where(grid["pct"].notna(), "—")
    cells.columns = [f"M{c}" for c in cells.columns]
    cells.insert(0, "Size", grid["size"])
    cells.index = cells.index.strftime("%b %Y")
    cells.index.name = "Cohort"
    sty = (cells.style
           .background_gradient(cmap=cmap, subset=cells.columns[1:], gmap=grid["pct"].to_numpy(),
                                axis=None, vmin=0, vmax=100)
           .set_properties(subset=cells.columns[1:], **{"text-align": "center", "white-space": "nowrap"}))
    return sty.set_caption(caption) if caption else sty


def _is_dark(rgb):
    lin = [c / 12.92 if c <= 0.04045 else ((c + 0.055) / 1.055) ** 2.4 for c in rgb]
    return 0.2126 * lin[0] + 0.7152 * lin[1] + 0.0722 * lin[2] < 0.408


def write_split_sheet(xw, name, grid):
    """Excel view of a split cohort: big line + small grey-size line in the same cell."""
    ws = xw.book.create_sheet(name)
    main, sub, pct, size = grid["main"], grid["sub"], grid["pct"], grid["size"]

    headers = ["Cohort", "Size"] + [f"M{c}" for c in pct.columns]
    for j, h in enumerate(headers, 1):
        cell = ws.cell(1, j, h)
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal="center")

    for i, cohort in enumerate(pct.index, 2):
        label_cell = ws.cell(i, 1, cohort.strftime("%b %Y"))
        label_cell.font = Font(bold=True)
        label_cell.alignment = Alignment(vertical="center")
        ws.cell(i, 2, int(size[cohort])).alignment = Alignment(horizontal="center", vertical="center")
        for j, idx in enumerate(pct.columns, 3):
            cell = ws.cell(i, j)
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
            p = pct.loc[cohort, idx]
            if pd.isna(p):
                cell.value = "—"
                continue
            rgb = cmap(p / 100)[:3]
            txt = "FFFFFF" if _is_dark(rgb) else "000000"
            cell.fill = PatternFill("solid", fgColor=mpl.colors.rgb2hex(rgb)[1:].upper())
            cell.value = CellRichText(
                TextBlock(InlineFont(rFont="Calibri", sz=11, b=True, color=txt), main.loc[cohort, idx] + "\n"),
                TextBlock(InlineFont(rFont="Calibri", sz=8, color=txt), sub.loc[cohort, idx]),
            )
        ws.row_dimensions[i].height = 32

    ws.column_dimensions["A"].width = 11
    ws.column_dimensions["B"].width = 8
    for j in range(3, len(headers) + 1):
        ws.column_dimensions[get_column_letter(j)].width = 24
    ws.freeze_panes = "C2"


def label(key):
    return " - ".join(map(str, key)) if isinstance(key, tuple) else str(key)


def cohorts_by(act, cols):
    tables, overall, clients = {}, {}, {}
    for key, grp in act.groupby(cols):
        key = key[0] if len(cols) == 1 else key
        styled, overall[key] = build_cohort(grp)
        clients[key] = grp["ClientID"].nunique()
        tables[key] = styled.set_caption(f"{label(key)}: {clients[key]:,} clients")

    summary = pd.DataFrame.from_dict(overall, orient="index").round(0)
    summary.columns = [f"M{c}" for c in summary.columns]
    summary.insert(0, "Clients", pd.Series(clients))
    summary.index.names = cols
    summary = summary.sort_values("Clients", ascending=False) if len(cols) == 1 else summary.sort_index()
    return tables, summary


def money_summary(ledger, client_ids):
    sub = ledger[ledger["ClientID"].isin(client_ids)]
    lt = sub["LedgerType"].str.strip().str.upper()
    money = (
        pd.DataFrame({
            "ClientID": sub["ClientID"],
            "total_deposit": sub["Credit"].where(lt == "DEPOSIT", 0),
            "total_withdraw": sub["Debit"].where(lt == "WITHDRAW", 0)
                            - sub["Credit"].where(lt == "WITHDRAW CANCELLED", 0),
        })
        .groupby("ClientID")[["total_deposit", "total_withdraw"]]
        .sum()
    )
    money["net_deposit"] = money["total_deposit"] - money["total_withdraw"]
    money["last_balance"] = sub.sort_values("CreatedAt").groupby("ClientID")["Balance"].last()
    return money


def sheet_name(name, used):
    base = re.sub(r"[\[\]:*?/\\]", "_", str(name)).strip("' ")[:31] or "Blank"
    name, i = base, 2
    while name.lower() in used:                      # Excel sheet names must be unique, ignoring case
        name = f"{base[:28]}_{i}"
        i += 1
    used.add(name.lower())
    return name

## 5. Overall cohort with Migrated / Fresh breakdown

In [27]:
all_grid = split_cohort(activity_source)
show_split(all_grid)

,Size,M0,M1,M2,M3,M4,M5,M6,M7,M8,M9,M10,M11,M12,M13
Cohort,,,,,,,,,,,,,,,
Sep 2025,719,719 (100%)Migrated: 418 · Fresh: 296 · Unmatched: 5,492 (68%)Migrated: 317 · Fresh: 172 · Unmatched: 3,313 (44%)Migrated: 223 · Fresh: 88 · Unmatched: 2,288 (40%)Migrated: 205 · Fresh: 81 · Unmatched: 2,276 (38%)Migrated: 192 · Fresh: 83 · Unmatched: 1,238 (33%)Migrated: 169 · Fresh: 67 · Unmatched: 2,217 (30%)Migrated: 158 · Fresh: 57 · Unmatched: 2,200 (28%)Migrated: 151 · Fresh: 47 · Unmatched: 2,207 (29%)Migrated: 151 · Fresh: 55 · Unmatched: 1,218 (30%)Migrated: 164 · Fresh: 53 · Unmatched: 1,210 (29%)Migrated: 162 · Fresh: 47 · Unmatched: 1,202 (28%)Migrated: 147 · Fresh: 55,196 (27%)Migrated: 140 · Fresh: 56,165 (23%)Migrated: 121 · Fresh: 44
Oct 2025,3125,"3,125 (100%)Migrated: 1,539 · Fresh: 1,586","1,711 (55%)Migrated: 963 · Fresh: 748","1,382 (44%)Migrated: 832 · Fresh: 550","1,283 (41%)Migrated: 808 · Fresh: 475","1,141 (37%)Migrated: 756 · Fresh: 385","1,051 (34%)Migrated: 708 · Fresh: 343",953 (30%)Migrated: 645 · Fresh: 308,903 (29%)Migrated: 620 · Fresh: 283,905 (29%)Migrated: 629 · Fresh: 276,887 (28%)Migrated: 616 · Fresh: 271,827 (26%)Migrated: 572 · Fresh: 255,748 (24%)Migrated: 520 · Fresh: 228,670 (21%)Migrated: 477 · Fresh: 193,—
Nov 2025,2624,"2,624 (100%)Migrated: 1,050 · Fresh: 1,574","1,427 (54%)Migrated: 645 · Fresh: 782","1,116 (43%)Migrated: 525 · Fresh: 591",934 (36%)Migrated: 464 · Fresh: 470,859 (33%)Migrated: 450 · Fresh: 409,736 (28%)Migrated: 406 · Fresh: 330,703 (27%)Migrated: 396 · Fresh: 307,731 (28%)Migrated: 409 · Fresh: 322,668 (25%)Migrated: 372 · Fresh: 296,619 (24%)Migrated: 355 · Fresh: 264,547 (21%)Migrated: 323 · Fresh: 224,464 (18%)Migrated: 260 · Fresh: 204,—,—
Dec 2025,2529,"2,529 (100%)Migrated: 956 · Fresh: 1,572 · Unmatched: 1","1,514 (60%)Migrated: 631 · Fresh: 882 · Unmatched: 1","1,100 (43%)Migrated: 501 · Fresh: 599",943 (37%)Migrated: 432 · Fresh: 511,797 (32%)Migrated: 405 · Fresh: 392,705 (28%)Migrated: 367 · Fresh: 338,739 (29%)Migrated: 385 · Fresh: 354,696 (28%)Migrated: 375 · Fresh: 321,621 (25%)Migrated: 340 · Fresh: 281,578 (23%)Migrated: 316 · Fresh: 262,512 (20%)Migrated: 272 · Fresh: 240,—,—,—
Jan 2026,2870,"2,870 (100%)Migrated: 1,016 · Fresh: 1,852 · Unmatched: 2","1,716 (60%)Migrated: 644 · Fresh: 1,072","1,231 (43%)Migrated: 518 · Fresh: 713",989 (34%)Migrated: 440 · Fresh: 548 · Unmatched: 1,869 (30%)Migrated: 412 · Fresh: 457,867 (30%)Migrated: 399 · Fresh: 468,793 (28%)Migrated: 376 · Fresh: 417,765 (27%)Migrated: 382 · Fresh: 383,705 (25%)Migrated: 361 · Fresh: 344,615 (21%)Migrated: 320 · Fresh: 295,—,—,—,—
Feb 2026,2723,"2,723 (100%)Migrated: 1,008 · Fresh: 1,713 · Unmatched: 2","1,619 (59%)Migrated: 682 · Fresh: 937","1,148 (42%)Migrated: 558 · Fresh: 590","1,033 (38%)Migrated: 515 · Fresh: 518",978 (36%)Migrated: 489 · Fresh: 489,872 (32%)Migrated: 447 · Fresh: 425,820 (30%)Migrated: 432 · Fresh: 388,742 (27%)Migrated: 413 · Fresh: 329,668 (25%)Migrated: 380 · Fresh: 288,—,—,—,—,—
Mar 2026,2483,"2,483 (100%)Migrated: 816 · Fresh: 1,664 · Unmatched: 3","1,346 (54%)Migrated: 487 · Fresh: 857 · Unmatched: 2",963 (39%)Migrated: 379 · Fresh: 584,854 (34%)Migrated: 337 · Fresh: 516 · Unmatched: 1,789 (32%)Migrated: 329 · Fresh: 460,689 (28%)Migrated: 297 · Fresh: 391 · Unmatched: 1,584 (24%)Migrated: 266 · Fresh: 317 · Unmatched: 1,523 (21%)Migrated: 242 · Fresh: 281,—,—,—,—,—,—
Apr 2026,2738,"2,738 (100%)Migrated: 886 · Fresh: 1,849 · Unmatched: 3","1,498 (55%)Migrated: 581 · Fresh: 916 · Unmatched: 1","1,134 (41%)Migrated: 516 · Fresh: 618",949 (35%)Migrated: 431 · Fresh: 518,856 (31%)Migrated: 411 · Fresh: 445,753 (28%)Migrated: 372 · Fresh: 381,649 (24%)Migrated: 338 · Fresh: 311,—,—,—,—,—,—,—
May 2026,2026,"2,026 (100%)Migrated: 633 · Fresh: 1,371 · Unmatched: 22","1,174 (58%)Migrated: 407 · Fresh: 766 · Unmatched: 1",815 (40%)Migrated: 323 · Fresh: 492,682 (34%)Migrated: 278 · Fresh: 404,599 (30%)Migrated: 250 · Fresh: 349,469 (23%)Migrated: 216 · Fresh: 253,—,

## 6. Clients who haven't billed this month

In [51]:
prev_ids = activity.loc[activity["Month"] == prev_month, "ClientID"]
this_ids = activity.loc[activity["Month"] == this_month, "ClientID"]
missing = prev_ids[~prev_ids.isin(this_ids)]

not_returned = (
    led[led["ClientID"].isin(missing)]
    .assign(in_prev=lambda d: d["Month"] == prev_month)
    .groupby("ClientID")
    .agg(cohort=("Month", "min"),
         bills_prev_month=("in_prev", "sum"),
         last_bill=("CreatedAt", "max"))
    .join(money_summary(ledger, missing))
    .sort_values("bills_prev_month", ascending=False)
)
print(f"{len(not_returned)} of {len(prev_ids)} {prev_month.strftime('%b %Y')} traders "
      f"have no BILL in {this_month.strftime('%b %Y')}")
not_returned

4817 of 11561 Sep 2026 traders have no BILL in Oct 2026


,cohort,bills_prev_month,last_bill,total_deposit,total_withdraw,net_deposit,last_balance
ClientID,,,,,,,
S118708,2026-05,1473,2026-09-25 15:05:36,583900.00,1048000.0,-464100.00,68.94
V22993,2025-10,1410,2026-09-24 14:39:58,3640831.00,6190039.0,-2549208.00,32.38
T124097,2026-05,1257,2026-09-23 19:40:37,66000.00,126700.0,-60700.00,72.14
V99922,2026-05,1104,2026-09-19 13:12:05,122105.00,4323.0,117782.00,264.67
P21328,2026-08,1089,2026-09-19 15:05:44,45536.77,32404.0,13132.77,0.36
...,...,...,...,...,...,...,...
S93473,2026-05,1,2026-09-19 18:44:00,41000.00,1146.0,39854.00,0.60
Y178878,2026-09,1,2026-09-30 14:23:46,1000.00,0.0,1000.00,133.00
Y175204,2026-09,1,2026-09-19 10:03:02,1000.00,1136.0,-136.00,0.80


In [52]:
inactive = (
    activity[(activity["Month"] < this_month) & ~activity["ClientID"].isin(this_ids)]
    .groupby("ClientID")
    .agg(cohort=("Month", "min"),
         last_active_month=("Month", "max"),
         active_months=("Month", "nunique"))
)
inactive["months_since_last_bill"] = (this_month - inactive["last_active_month"]).apply(lambda x: x.n)
inactive["last_bill"] = led.groupby("ClientID")["CreatedAt"].max()
inactive = (inactive.join(money_summary(ledger, inactive.index))
            .sort_values(["months_since_last_bill", "last_balance"], ascending=[True, False]))

ladder = inactive.groupby("months_since_last_bill").agg(
    clients=("cohort", "size"),
    net_deposit=("net_deposit", "sum"),
    balance_left=("last_balance", "sum"),
)
ladder

,clients,net_deposit,balance_left
months_since_last_bill,,,
1,4817,91867673.37,1.485463e+07
2,3070,70076876.71,1.012698e+07
3,2484,27517099.58,1.974535e+06
4,2210,34980606.18,4.581569e+06
5,1673,-7947903.18,3.199951e+07
6,1848,8677606.48,1.438461e+06
7,1763,20191565.81,5.818364e+05
8,1598,-9904292.64,5.231070e+05
9,1421,7257163.78,3.154055e+05


## 7. Cohorts by source (with Migrated / Fresh breakdown) and retention summaries

In [54]:
_, source_summary = cohorts_by(activity_source, [source_col])
_, status_summary = cohorts_by(activity_source, ["Client Status"])
_, combo_summary = cohorts_by(activity_source, [source_col, "Client Status"])

source_grids = {src: split_cohort(grp) for src, grp in activity_source.groupby(source_col)}

display(source_summary, status_summary, combo_summary)

,Clients,M0,M1,M2,M3,M4,M5,M6,M7,M8,M9,M10,M11,M12,M13
Source One,,,,,,,,,,,,,,,
MAIN WEBSITE (HOT),23105,100.0,54.0,42.0,37.0,34.0,31.0,30.0,29.0,28.0,26.0,25.0,24.0,24.0,23.0
Referral (HOT),7085,100.0,58.0,41.0,34.0,31.0,28.0,24.0,23.0,23.0,21.0,20.0,19.0,16.0,25.0
META6,2345,100.0,47.0,31.0,26.0,23.0,21.0,19.0,19.0,17.0,18.0,18.0,17.0,NaN,NaN
Telegram,466,100.0,16.0,7.0,6.0,3.0,2.0,1.0,1.0,1.0,2.0,1.0,0.0,0.0,NaN
Telegram Ads - Aman,304,100.0,47.0,32.0,29.0,33.0,38.0,37.0,31.0,29.0,32.0,38.0,8.0,NaN,NaN
Unknown,87,100.0,28.0,16.0,10.0,3.0,8.0,19.0,15.0,10.0,12.0,17.0,0.0,0.0,0.0
RCS,79,100.0,57.0,51.0,58.0,47.0,44.0,48.0,41.0,48.0,34.0,35.0,35.0,NaN,NaN
Meta(Murtaza-E),51,100.0,37.0,25.0,20.0,20.0,16.0,15.0,17.0,NaN,NaN,NaN,NaN,NaN,NaN
Instagram,47,100.0,43.0,26.0,15.0,13.0,11.0,17.0,25.0,24.0,21.0,30.0,10.0,NaN,NaN


,Clients,M0,M1,M2,M3,M4,M5,M6,M7,M8,M9,M10,M11,M12,M13
Client Status,,,,,,,,,,,,,,,
Fresh,20034,100.0,53.0,35.0,29.0,25.0,22.0,20.0,19.0,18.0,17.0,15.0,14.0,13.0,15.0
Migrated,13717,100.0,55.0,48.0,45.0,43.0,41.0,39.0,38.0,37.0,36.0,34.0,31.0,32.0,29.0
Unmatched,50,100.0,23.0,7.0,10.0,3.0,8.0,19.0,15.0,10.0,12.0,17.0,0.0,0.0,0.0


Clients     M0     M1     M2     M3  \
Source One            Client Status                                        
APPLICATION           Fresh                1  100.0  100.0  100.0  100.0   
                      Migrated             1  100.0  100.0  100.0    0.0   
ARK ACTIVE            Fresh                8  100.0    NaN    NaN    NaN   
                      Migrated             2  100.0    NaN    NaN    NaN   
IGL                   Fresh                1  100.0  100.0  100.0    NaN   
Instagram             Fresh               40  100.0   38.0   22.0   12.0   
                      Migrated             7  100.0   83.0   50.0   33.0   
MAIN WEBSITE          Fresh                1  100.0    0.0    NaN    NaN   
                      Migrated             1  100.0  100.0  100.0  100.0   
MAIN WEBSITE (HOT)    Fresh            10655  100.0   53.0   35.0   30.0   
                      Migrated         12450  100.0   55.0   48.0   45.0   
MAIN WEBSITE (HOT) 1  Fresh               15  100.0   56.0   25.0    0.0   
                      Migrated             2  100.0   50.0    NaN    NaN   
MAIN WEBSITE (HOT) C  Fresh               19  100.0   81.0   36.0   40.0   
                      Migrated             9  100.0   75.0   62.0   71.0   
META-E                Fresh                2  100.0  100.0  100.0    NaN   
                      Migrated             1  100.0  100.0  100.0    0.0   
META2                 Migrated             1  100.0  100.0  100.0  100.0   
META3                 Fresh               10  100.0   57.0   33.0    0.0   
                      Migrated             1  100.0  100.0    0.0  100.0   
META6                 Fresh             2165  100.0   47.0   30.0   25.0   
                      Migrated           180  100.0   49.0   38.0   32.0   
Meta (Murtaza)        Fresh               16  100.0   62.0   50.0   31.0   
                      Migrated             2  100.0   50.0   50.0   50.0   
Meta Crypto           Migrated             1  100.0  100.0  100.0    0.0   
Meta Lookalike        Fresh                8  100.0   62.0   43.0   43.0   
                      Migrated             4  100.0   25.0   25.0   25.0   
Meta ads (Jay) - Aman Fresh                1  100.0    NaN    NaN    NaN   
Meta(Digirover)       Fresh                1  100.0    0.0    0.0    0.0   
Meta(Murtaza-E)       Fresh               49  100.0   37.0   27.0   20.0   
                      Migrated             2  100.0   50.0    0.0    0.0   
Meta(Murtaza2)        Fresh               21  100.0   48.0   33.0   25.0   
                      Migrated             3  100.0   67.0   33.0    0.0   
Meta-IMR              Fresh                2  100.0  100.0    NaN    NaN   
Meta-K                Migrated             1  100.0    0.0  100.0    NaN   
Meta-Saudi            Fresh                4  100.0   75.0   50.0   50.0   
Meta55 - Aman         Fresh                3  100.0   50.0    0.0    NaN   
PRS                   Fresh                1  100.0  100.0  100.0  100.0   
RCS                   Fresh                8  100.0   38.0   57.0   43.0   
                      Migrated            71  100.0   59.0   51.0   59.0   
Referral (HOT)        Fresh             6279  100.0   57.0   39.0   32.0   
                      Migrated           806  100.0   67.0   51.0   46.0   
STOCKMEME             Fresh               35  100.0   43.0   23.0   17.0   
                      Migrated             4  100.0   25.0    0.0    0.0   
Sanchit_Data          Fresh               10  100.0    NaN    NaN    NaN   
TG2709                Fresh                2  100.0   50.0   50.0   50.0   
Telegram              Fresh              394  100.0   17.0    6.0    5.0   
                      Migrated            72  100.0   10.0   11.0    8.0   
Telegram Ads - Aman   Fresh              229  100.0   43.0   24.0   19.0   
                      Migrated            75  100.0   57.0   50.0   47.0   
Uncategorized         Fresh                1  100.0  100.0  100.0  100.0   
         

In [55]:
# view any one source's cohort, e.g. the biggest one
top_source = source_summary.index[0]
show_split(source_grids[top_source], caption=top_source)

,Size,M0,M1,M2,M3,M4,M5,M6,M7,M8,M9,M10,M11,M12,M13
Cohort,,,,,,,,,,,,,,,
Sep 2025,687,687 (100%)Migrated: 399 · Fresh: 288,466 (68%)Migrated: 301 · Fresh: 165,298 (43%)Migrated: 212 · Fresh: 86,274 (40%)Migrated: 196 · Fresh: 78,263 (38%)Migrated: 182 · Fresh: 81,224 (33%)Migrated: 158 · Fresh: 66,207 (30%)Migrated: 151 · Fresh: 56,188 (27%)Migrated: 143 · Fresh: 45,198 (29%)Migrated: 145 · Fresh: 53,207 (30%)Migrated: 156 · Fresh: 51,201 (29%)Migrated: 155 · Fresh: 46,194 (28%)Migrated: 141 · Fresh: 53,190 (28%)Migrated: 135 · Fresh: 55,159 (23%)Migrated: 117 · Fresh: 42
Oct 2025,2678,"2,678 (100%)Migrated: 1,428 · Fresh: 1,250","1,467 (55%)Migrated: 890 · Fresh: 577","1,199 (45%)Migrated: 769 · Fresh: 430","1,108 (41%)Migrated: 746 · Fresh: 362",992 (37%)Migrated: 698 · Fresh: 294,931 (35%)Migrated: 658 · Fresh: 273,852 (32%)Migrated: 605 · Fresh: 247,806 (30%)Migrated: 582 · Fresh: 224,806 (30%)Migrated: 594 · Fresh: 212,792 (30%)Migrated: 581 · Fresh: 211,732 (27%)Migrated: 536 · Fresh: 196,668 (25%)Migrated: 488 · Fresh: 180,605 (23%)Migrated: 448 · Fresh: 157,—
Nov 2025,1751,"1,751 (100%)Migrated: 869 · Fresh: 882","1,061 (61%)Migrated: 559 · Fresh: 502",816 (47%)Migrated: 454 · Fresh: 362,685 (39%)Migrated: 398 · Fresh: 287,635 (36%)Migrated: 390 · Fresh: 245,555 (32%)Migrated: 359 · Fresh: 196,538 (31%)Migrated: 352 · Fresh: 186,560 (32%)Migrated: 361 · Fresh: 199,504 (29%)Migrated: 330 · Fresh: 174,474 (27%)Migrated: 315 · Fresh: 159,409 (23%)Migrated: 280 · Fresh: 129,345 (20%)Migrated: 225 · Fresh: 120,—,—
Dec 2025,1783,"1,783 (100%)Migrated: 843 · Fresh: 940","1,099 (62%)Migrated: 550 · Fresh: 549",808 (45%)Migrated: 439 · Fresh: 369,697 (39%)Migrated: 380 · Fresh: 317,599 (34%)Migrated: 361 · Fresh: 238,531 (30%)Migrated: 324 · Fresh: 207,561 (31%)Migrated: 340 · Fresh: 221,532 (30%)Migrated: 333 · Fresh: 199,466 (26%)Migrated: 298 · Fresh: 168,430 (24%)Migrated: 282 · Fresh: 148,381 (21%)Migrated: 240 · Fresh: 141,—,—,—
Jan 2026,1907,"1,907 (100%)Migrated: 865 · Fresh: 1,042","1,160 (61%)Migrated: 546 · Fresh: 614",839 (44%)Migrated: 442 · Fresh: 397,694 (36%)Migrated: 375 · Fresh: 319,613 (32%)Migrated: 358 · Fresh: 255,608 (32%)Migrated: 348 · Fresh: 260,571 (30%)Migrated: 327 · Fresh: 244,546 (29%)Migrated: 329 · Fresh: 217,498 (26%)Migrated: 315 · Fresh: 183,427 (22%)Migrated: 272 · Fresh: 155,—,—,—,—
Feb 2026,1794,"1,794 (100%)Migrated: 908 · Fresh: 886","1,112 (62%)Migrated: 621 · Fresh: 491",816 (45%)Migrated: 510 · Fresh: 306,734 (41%)Migrated: 473 · Fresh: 261,693 (39%)Migrated: 445 · Fresh: 248,601 (34%)Migrated: 405 · Fresh: 196,574 (32%)Migrated: 396 · Fresh: 178,536 (30%)Migrated: 376 · Fresh: 160,483 (27%)Migrated: 349 · Fresh: 134,—,—,—,—,—
Mar 2026,1562,"1,562 (100%)Migrated: 709 · Fresh: 853",889 (57%)Migrated: 427 · Fresh: 462,649 (42%)Migrated: 330 · Fresh: 319,584 (37%)Migrated: 296 · Fresh: 288,550 (35%)Migrated: 294 · Fresh: 256,468 (30%)Migrated: 263 · Fresh: 205,403 (26%)Migrated: 235 · Fresh: 168,356 (23%)Migrated: 211 · Fresh: 145,—,—,—,—,—,—
Apr 2026,1618,"1,618 (100%)Migrated: 772 · Fresh: 846",940 (58%)Migrated: 514 · Fresh: 426,751 (46%)Migrated: 460 · Fresh: 291,629 (39%)Migrated: 383 · Fresh: 246,578 (36%)Migrated: 366 · Fresh: 212,516 (32%)Migrated: 331 · Fresh: 185,460 (28%)Migrated: 302 · Fresh: 158,—,—,—,—,—,—,—
May 2026,1266,"1,266 (100%)Migrated: 541 · Fresh: 725",741 (59%)Migrated: 345 · Fresh: 396,528 (42%)Migrated: 278 · Fresh: 250,447 (35%)Migrated: 233 · Fresh: 214,385 (30%)Migrated: 205 · Fresh: 180,300 (24%)Migrated: 178 · Fresh: 122,—,—,—,—,—,—,—,—


## 8. Export all cohorts to one workbook

In [57]:
os.makedirs(COHORT_DIR, exist_ok=True)
file_path = os.path.join(COHORT_DIR, f"cohorts_{last_month}.xlsx")
used = set()

with pd.ExcelWriter(file_path, engine="openpyxl") as xw:
    source_summary.to_excel(xw, sheet_name=sheet_name("Summary - Source", used))
    status_summary.to_excel(xw, sheet_name=sheet_name("Summary - Status", used))
    combo_summary.to_excel(xw, sheet_name=sheet_name("Summary - Source x Status", used))
    write_split_sheet(xw, sheet_name("All Clients", used), all_grid)
    for src in source_summary.index:                 # biggest sources first
        write_split_sheet(xw, sheet_name(src, used), source_grids[src])

print("Saved:", file_path)
os.startfile(COHORT_DIR)

Saved: C:\Users\Administrator\Videos\co\cohorts_2026-10.xlsx


In [59]:
chat = pd.read_csv(r"C:\Users\Administrator\Videos\Chat\History (11).csv", skiprows=2, header=0)
chat

C:\Users\Administrator\AppData\Local\Temp\ipykernel_10476\810819595.py:1: DtypeWarning: Columns (5: First Name) have mixed types. Specify dtype option on import or set low_memory=False.
  chat = pd.read_csv(r"C:\Users\Administrator\Videos\Chat\History (11).csv", skiprows=2, header=0)


,S.No,Visitor ID,Conversation ID,Name,Salutation,First Name,Last Name,Email Address,Phone Number,Country/Region,...,System Event Count,Info Message Count,Initiated By,Ended By,Missed Reason,Last Email Sent Time,Last Email Sent Time(in millis),Customer Info,Tags,Chat Initiated URL
0,1,#33394,1174320000000604048,Shivam yadav,-,NaN,Shivam yadav,sy217161@gamil.com,NaN,India,...,0.0,0.0,bot,bot,NaN,NaN,NaN,"{""Client_Name"":""Shivam yadav"",""Email"":""SY21716...",NaN,https://web.tradex56.live/support
1,2,#33393,1174320000000626034,Hundekoti Nageshwari,-,NaN,Hundekoti Nageshwari,dandu.nageshwari@gmail.com,NaN,India,...,0.0,0.0,visitor,bot,NaN,NaN,NaN,"{""Client_Name"":""Hundekoti Nageshwari"",""Email"":...",NaN,https://web.tradex-55.live/support
2,3,#33392,1174320000000610044,Hundekoti Nageshwari,-,NaN,Hundekoti Nageshwari,dandu.nageshwari@gmail.com,NaN,India,...,0.0,0.0,visitor,bot,NaN,NaN,NaN,"{""Client_Name"":""Hundekoti Nageshwari"",""Email"":...",NaN,https://web.tradex-55.live/support
3,4,#33391,1174320000000610043,Nanda kumar,-,NaN,Nanda kumar,nanduvibgyor@gmail.com,NaN,India,...,0.0,3.0,visitor,agent,NaN,NaN,NaN,"{""Client_Name"":""Nanda kumar"",""Email"":""NANDUVIB...","Delay deposit > 10 min,New Platform",https://web.tradex-55.live/support
4,5,#33390,1174320000000602041,Ishtiyaq Ahmed,-,NaN,Ishtiyaq Ahmed,ishtiyaq.abbasi@gmail.com,NaN,India,...,0.0,3.0,visitor,agent,NaN,NaN,NaN,"{""client_id"":""I212161""}","Delay withdrawal > 45 min,New Platform",MainActivity
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33382,33383,#6,1174320000000023002,Visitor 64726,-,NaN,Visitor 64726,NaN,NaN,India,...,0.0,2.0,visitor,agent,NaN,NaN,NaN,NaN,NaN,https://www.w3schools.com/html/tryit.asp?filen...
33383,33384,#5,1174320000000018003,Visitor 20207,-,NaN,Visitor 20207,NaN,NaN,India,...,0.0,2.0,visitor,agent,NaN,NaN,NaN,NaN,NaN,https://www.w3schools.com/html/tryit.asp?filen...
33384,33385,#4,1174320000000021001,Visitor 59051,-,NaN,Visitor 59051,NaN,NaN,India,...,0.0,2.0,visitor,agent,NaN,NaN,NaN,NaN,NaN,https://www.w3schools.com/html/tryit.asp?filen...
33385,33386,#3,1174320000000017007,Visitor 59051,-,NaN,Visitor 59051,NaN,NaN,India,...,0.0,1.0,visitor,visitor,NaN,NaN,NaN,NaN,NaN,https://www.w3schools.com/html/tryit.asp?filen...


In [60]:
details = chat["Customer Info"].apply(
    lambda x: json.loads(x) if isinstance(x, str) else {}
)

details_df = pd.json_normalize(details)

details_df = pd.concat(
    [
        chat[["Created Time", "Tags"]].reset_index(drop=True),
        details_df.reset_index(drop=True)
    ],
    axis=1
)

details_df

,Created Time,Tags,Client_Name,Email,Lead_Source1,CM_Lead_Source_2,Visitor_Type,Lead_Source_2,Client_Id,CM_Lead_Source_1,Name,client_id,sourceId
0,22/10/2026 23:58:36,NaN,Shivam yadav,SY217161@GAMIL.COM,MAIN WEBSITE (HOT),6823819000006114001,logged_in,ChatBot,S52417,MAIN WEBSITE (HOT),Shivam yadav,NaN,NaN
1,22/10/2026 23:58:18,NaN,Hundekoti Nageshwari,DANDU.NAGESHWARI@GMAIL.COM,MAIN WEBSITE (HOT),6823819000006114001,logged_in,ChatBot,H73648,MAIN WEBSITE (HOT),Hundekoti Nageshwari,NaN,NaN
2,22/10/2026 23:56:43,NaN,Hundekoti Nageshwari,DANDU.NAGESHWARI@GMAIL.COM,MAIN WEBSITE (HOT),6823819000006114001,logged_in,ChatBot,H73648,MAIN WEBSITE (HOT),Hundekoti Nageshwari,NaN,NaN
3,22/10/2026 23:56:10,"Delay deposit > 10 min,New Platform",Nanda kumar,NANDUVIBGYOR@GMAIL.COM,MAIN WEBSITE (HOT),6823819000006114001,logged_in,ChatBot,N112901,MAIN WEBSITE (HOT),Nanda kumar,NaN,NaN
4,22/10/2026 23:54:41,"Delay withdrawal > 45 min,New Platform",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,I212161,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
33382,20/09/2026 17:25:16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33383,20/09/2026 17:03:47,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33384,20/09/2026 16:59:10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33385,20/09/2026 16:57:18,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [61]:
details_df = details_df[['Created Time', 'Tags', 'Client_Id']]
details_df

,Created Time,Tags,Client_Id
0,22/10/2026 23:58:36,NaN,S52417
1,22/10/2026 23:58:18,NaN,H73648
2,22/10/2026 23:56:43,NaN,H73648
3,22/10/2026 23:56:10,"Delay deposit > 10 min,New Platform",N112901
4,22/10/2026 23:54:41,"Delay withdrawal > 45 min,New Platform",NaN
...,...,...,...
33382,20/09/2026 17:25:16,NaN,NaN
33383,20/09/2026 17:03:47,NaN,NaN
33384,20/09/2026 16:59:10,NaN,NaN
33385,20/09/2026 16:57:18,NaN,NaN


In [62]:
details_df = details_df.dropna(subset=["Tags"])
details_df = details_df.dropna(subset=["Client_Id"])
details_df = details_df[details_df["Tags"].str.strip() != ""]
details_df

,Created Time,Tags,Client_Id
3,22/10/2026 23:56:10,"Delay deposit > 10 min,New Platform",N112901
9,22/10/2026 23:48:52,"Delay withdrawal > 45 min,New Platform",H196459
10,22/10/2026 23:45:19,"Withdraw,New Platform",S27951
13,22/10/2026 23:44:00,"Withdraw,New Platform",H176928
14,22/10/2026 23:42:27,"Adjustment,New Platform,Follow Up",P92022
...,...,...,...
33346,21/09/2026 04:29:09,"Glitches and Software Issue,New Platform",C20270
33360,21/09/2026 01:04:29,"Withdraw,New Platform",A98442
33369,21/09/2026 00:32:36,"Platform related,New Platform",A31411
33375,20/09/2026 23:37:00,"Platform related,New Platform",B125737


In [63]:
not_returned

,cohort,bills_prev_month,last_bill,total_deposit,total_withdraw,net_deposit,last_balance
ClientID,,,,,,,
S118708,2026-05,1473,2026-09-25 15:05:36,583900.00,1048000.0,-464100.00,68.94
V22993,2025-10,1410,2026-09-24 14:39:58,3640831.00,6190039.0,-2549208.00,32.38
T124097,2026-05,1257,2026-09-23 19:40:37,66000.00,126700.0,-60700.00,72.14
V99922,2026-05,1104,2026-09-19 13:12:05,122105.00,4323.0,117782.00,264.67
P21328,2026-08,1089,2026-09-19 15:05:44,45536.77,32404.0,13132.77,0.36
...,...,...,...,...,...,...,...
S93473,2026-05,1,2026-09-19 18:44:00,41000.00,1146.0,39854.00,0.60
Y178878,2026-09,1,2026-09-30 14:23:46,1000.00,0.0,1000.00,133.00
Y175204,2026-09,1,2026-09-19 10:03:02,1000.00,1136.0,-136.00,0.80


In [66]:
# Make sure Created Time is a datetime
details_df["Created Time"] = pd.to_datetime(
    details_df["Created Time"],
    dayfirst=True,
    errors="coerce"
)

# Format the chat details into one column
details_df["Chat Details"] = (
      details_df["Created Time"].dt.strftime("%d %b %Y, %H:%M")
     + details_df["Tags"].fillna("No tags")
)

# Match ClientID in not_returned with Client_Id in details_df
chat_lookup = details_df.groupby("Client_Id")["Chat Details"].apply(
    lambda x: "\n\n".join(x)
)

# Add the new column to not_returned
not_returned["Chat Details"] = (
    not_returned.index.to_series()
    .map(chat_lookup)
    .fillna("No matching chat")
)

not_returned

,cohort,bills_prev_month,last_bill,total_deposit,total_withdraw,net_deposit,last_balance,Chat Details
ClientID,,,,,,,,
S118708,2026-05,1473,2026-09-25 15:05:36,583900.00,1048000.0,-464100.00,68.94,"24 Sep 2026, 14:47Deposit,New Platform\n\n24 S..."
V22993,2025-10,1410,2026-09-24 14:39:58,3640831.00,6190039.0,-2549208.00,32.38,"22 Sep 2026, 15:17Glitches and Software Issue,..."
T124097,2026-05,1257,2026-09-23 19:40:37,66000.00,126700.0,-60700.00,72.14,No matching chat
V99922,2026-05,1104,2026-09-19 13:12:05,122105.00,4323.0,117782.00,264.67,"09 Oct 2026, 15:48Deposit,New Platform"
P21328,2026-08,1089,2026-09-19 15:05:44,45536.77,32404.0,13132.77,0.36,No matching chat
...,...,...,...,...,...,...,...,...
S93473,2026-05,1,2026-09-19 18:44:00,41000.00,1146.0,39854.00,0.60,No matching chat
Y178878,2026-09,1,2026-09-30 14:23:46,1000.00,0.0,1000.00,133.00,No matching chat
Y175204,2026-09,1,2026-09-19 10:03:02,1000.00,1136.0,-136.00,0.80,No matching chat


In [68]:
os.makedirs(NOT_RETURNED_DIR, exist_ok=True)
file_path = os.path.join(NOT_RETURNED_DIR, f"inactive_clients_{this_month}.xlsx")

with pd.ExcelWriter(file_path) as xw:
    not_returned.astype({"cohort": str}).to_excel(xw, sheet_name="Not returned last month")
    inactive.astype({"cohort": str, "last_active_month": str}).to_excel(xw, sheet_name="All inactive")
    ladder.to_excel(xw, sheet_name="Summary")

print("Saved:", file_path)

Saved: C:\Users\Administrator\Videos\Not Returned\inactive_clients_2026-10.xlsx
